# Bose2: AI video + Egyptian Arabic voice trial

Manual experiment only. This notebook tests one short generated scene and adds Egyptian Arabic narration. It does not upload or publish. Use a GPU only if Colab offers it free; if not, stop. Do not enable billing or buy credits. Free GPU availability is not guaranteed.

In [ ]:
%pip -q install -U diffusers transformers accelerate imageio-ffmpeg edge-tts
import torch, shutil
if not torch.cuda.is_available(): raise RuntimeError('No free GPU available in this session. Stop; do not enable billing.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
if not shutil.which('ffmpeg'): raise RuntimeError('FFmpeg unavailable.')

In [ ]:
from pathlib import Path
import subprocess
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video
out = Path('/content/bose2_trial'); out.mkdir(exist_ok=True)
prompt = 'Cinematic macro shot of hands crocheting cream yarn with a crochet hook on a wooden table, warm natural window light, shallow depth of field, slow camera movement, realistic yarn texture, documentary craft film, stable framing, no text.'
script = 'النهارده هنتعلم بداية غرزة كروشيه بسيطة خطوة بخطوة.'
pipe = CogVideoXPipeline.from_pretrained('zai-org/CogVideoX-2b', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload(); pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
frames = pipe(prompt=prompt, num_videos_per_prompt=1, num_inference_steps=30, num_frames=49, guidance_scale=6, generator=torch.Generator(device='cuda').manual_seed(42)).frames[0]
silent = out / 'scene.mp4'; export_to_video(frames, str(silent), fps=8)
if not silent.exists() or silent.stat().st_size < 1024: raise RuntimeError('No valid video was generated.')
audio = out / 'voice.mp3'; final = out / 'bose2_ai_voice_video_test.mp4'
subprocess.run(['edge-tts','--voice','ar-EG-SalmaNeural','--rate=-5%','--text',script,'--write-media',str(audio)], check=True, timeout=90)
subprocess.run(['ffmpeg','-y','-i',str(silent),'-i',str(audio),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-shortest','-movflags','+faststart',str(final)], check=True, timeout=120)
if not final.exists() or final.stat().st_size < 1024: raise RuntimeError('Final video invalid.')
print('Generated:', final, 'bytes:', final.stat().st_size)
from IPython.display import Video, display; display(Video(str(final), embed=True))

In [ ]:
from google.colab import files
files.download('/content/bose2_trial/bose2_ai_voice_video_test.mp4')

## Review before production

Listen to the narration and inspect the hands, hook, yarn, motion, and flicker. This short experiment is not a finished lesson and does not prove that the model can accurately show a stitch. Production remains unchanged until the clip is generated and manually approved.